# Polymarket loss-chasing rough read

This notebook records the first-pass checks for 2024-01-01 through 2025-12-31 UTC. The 2026-01-01 through 2026-03-29 trade holdout stays outside the analysis.

Source documentation: https://www.vincentgregoire.com/polymarket-users-data/

In [ ]:
import json
from pathlib import Path
import polars as pl

OUTPUT = Path('analysis/output')
summary = json.loads((OUTPUT / 'summary.json').read_text())
episodes = pl.read_parquet(OUTPUT / 'loss-episodes.parquet')
wallets = pl.read_parquet(OUTPUT / 'wallet-metrics-with-control.parquet')
summary['sample'], summary['primary_signal']

In [ ]:
# Independent episode-level checks.
episodes.select([
    pl.len().alias('eligible_episodes'),
    (pl.col('post_loss_stake_usd') > 0).mean().alias('active_next_day_share'),
    pl.col('chasing_episode').cast(pl.Float64).mean().alias('chasing_episode_share'),
    pl.col('post_loss_ratio').filter(pl.col('post_loss_stake_usd') > 0).median().alias('active_ratio_median'),
    pl.col('post_loss_ratio').filter(pl.col('post_loss_stake_usd') > 0).quantile(0.75).alias('active_ratio_p75'),
])

In [ ]:
# The saved wallet result supports both denominators.
wallets.select([
    pl.len().alias('wallets_with_at_least_three_episodes'),
    pl.col('consistent_chasing_wallet').sum().alias('consistent_chasing_wallets'),
    pl.col('consistent_chasing_wallet').mean().alias('share_of_episode_eligible_wallets'),
])

Interpretation: the primary rule identifies a small repeated-chasing subgroup. The active next-day median sits near 1.00x baseline, so the result does not support a broad post-loss stake increase. The next pass should freeze this definition and test Jan-Mar 2026, then add an active-only control comparison.